> **H's checklist — DELETE THIS CELL before submitting**
> 1. Search for `<<` and replace every placeholder (names, GitHub user, read-outs, "why it failed" notes).
> 2. Check that `REPO_URL` / `REPO_SUBDIR` (setup cell) and the Colab badge link below point at the final public repo and folder.
> 3. After G3, run once with `OUTPUTS_DIR = "outputs"` and read every cell's output for errors or `GAP:` lines.
> 4. Section 7: choose 6 demo examples (2 per language), set `REFRESH_CACHE = True` once in a session where the model loads, then set it back to `False`.
> 5. *Runtime → Restart and run all* in a **fresh** Colab runtime. It must finish with no red errors. Then *File → Save* so outputs stay visible.
> 6. Share as "Anyone with the link can view" (or give the GitHub link). Open it in an incognito window to check.
> 7. To rehearse with fake data any time: `python tools/make_dummy.py`, then set `OUTPUTS_DIR = "outputs_dummy"`. Never submit a run on fake data.

# SafeLabel — can you trust a machine translation of a medicine label?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/auctacius/utkarshpearl/blob/main/safelabel/notebooks/SafeLabel_final.ipynb)

**Track:** AI + Languages · **Team:** <<names>> (Hindi · Chinese · Arabic)

> This project is a prototype and is not medical advice.

**In one sentence:** SafeLabel helps a family member who reads Hindi, Chinese or Arabic by taking an English medicine-label sentence and giving back a translation plus a warning — *don't rely on this* or *no problems found by our checks* — with the reason.

**The question we tested:** when a free translator (NLLB-200) translates medicine instructions, how often does a speaker of the language judge the result **unsafe**, and can simple checks warn a reader better than a normal quality score?

**How to run:** *Runtime → Run all*. Every section reads saved results, so no GPU is needed; a GPU only makes the optional live demo in section 7 faster.

**Contents:** 1 Setup · 2 Data · 3 Human ratings · 4 Detectors vs the do-nothing baseline · 5 Where it fails · 6 Quality score vs human verdict · 7 Demo · 8 How we produced the numbers · 9 Limits and next steps · 10 Credits

## 1. Setup
The first cell downloads our code from GitHub (only needed in a fresh Colab runtime). The second installs the few libraries we use.

In [ ]:
# 1. SETUP - get our project files.
# In plain language: the first time this runs in a fresh Colab, it downloads our code from GitHub.
import os, sys, json, html, subprocess
from pathlib import Path

REPO_URL = "https://github.com/auctacius/utkarshpearl"   # the team's public GitHub repo
REPO_SUBDIR = "safelabel"                                  # our project lives in this folder of that repo
OUTPUTS_DIR = "outputs"      # use "outputs_dummy" to rehearse this notebook with fake data (never submit that)

if not Path("data/sentences_en.csv").exists():             # we are not inside the project folder yet (fresh Colab)
    if "<<" in REPO_URL:
        raise RuntimeError("Set REPO_URL in this cell to the public GitHub repo, then run it again.")
    if not Path("repo").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "repo"], check=True)
    os.chdir(Path("repo") / REPO_SUBDIR)

sys.path.insert(0, str(Path("src").resolve()))    # lets us "import safelabel"
sys.path.insert(0, str(Path.cwd()))               # lets us "import app.app"
print("Working folder:", Path.cwd())
print("Reading results from:", OUTPUTS_DIR)

In [ ]:
%pip -q install -r requirements.txt

In [ ]:
# HELPERS - small functions that the later cells share.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

LANGS = ["hi", "zh", "ar"]
LANG_NAMES = {"hi": "Hindi", "zh": "Chinese", "ar": "Arabic", "all": "All three"}
DETECTOR_NAMES = {
    "always_safe": "Always says 'fine' (baseline)",
    "rt_chrf": "Round-trip chrF score",
    "labse": "LaBSE similarity score",
    "number_rule": "Number check",
    "negation_rule": "Negation check",
    "rules": "Number OR negation",
    "rules_plus_chrf": "Rules OR chrF score",
}
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})


def load_csv(name):
    """Read a CSV from the outputs folder as text; return None (and say so) if it is not there yet."""
    path = Path(OUTPUTS_DIR) / name
    if not path.exists():
        print(f"(not available yet: {path})")
        return None
    return pd.read_csv(path, encoding="utf-8-sig", dtype=str, keep_default_na=False)


def num(df, cols):
    """Turn text columns into numbers (a blank becomes NaN)."""
    df = df.copy()
    for c in cols:
        if c in df:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


TABLE_STYLE = ("<style>.sl td,.sl th{border:1px solid #888;padding:4px 8px;vertical-align:top;text-align:left}"
               ".sl{border-collapse:collapse}</style>")


def show_examples(df, per_lang=3, kind=None):
    """Show examples side by side: English / machine translation / correct translation.
    dir='auto' lets Arabic text run right-to-left."""
    if df is None or len(df) == 0:
        print("(no examples yet)")
        return
    rows = []
    for lang in LANGS:
        sub = df[df["lang"] == lang]
        if kind:
            sub = sub[sub["kind"] == kind]
        for _, r in sub.head(per_lang).iterrows():
            tag = f"{r['sentence_id']} · {r.get('kind', '')} · {r.get('error_type', '')}"
            rows.append(
                f"<tr><td>{LANG_NAMES[lang]}<br><small>{html.escape(tag)}</small></td>"
                f"<td dir='auto'>{html.escape(r['src_en'])}</td>"
                f"<td dir='auto'>{html.escape(r['translation'])}</td>"
                f"<td dir='auto'>{html.escape(r.get('corrected_translation', '') or '—')}</td></tr>")
    head = "<tr><th>Language</th><th>English</th><th>Machine translation</th><th>Correct (checked by a speaker)</th></tr>"
    display(HTML(TABLE_STYLE + "<table class='sl'>" + head + "".join(rows) + "</table>"))


BANNER = {
    "dont_rely": ("#C8453B", "Don't rely on this translation"),
    "unsure": ("#B7791F", "Unsure - check with your pharmacist"),
    "looks_consistent": ("#2F7D5B", "No problems found by our checks - still confirm with your pharmacist"),
}


def show_result(r):
    """Show one result of check_translation() as a coloured box. We never say a translation is 'safe'."""
    color, title = BANNER.get(r["verdict"], ("#555555", r["verdict"]))
    reasons = "".join(f"<li>{html.escape(x)}</li>" for x in r.get("reasons", []))
    display(HTML(
        f"<div style='border-left:8px solid {color};padding:8px 12px;margin:8px 0'>"
        f"<b style='color:{color}'>{html.escape(title)}</b>"
        f"<div><i>English:</i> {html.escape(r['src_en'])}</div>"
        f"<div dir='auto'><i>{LANG_NAMES.get(r['lang'], r['lang'])}:</i> {html.escape(r['translation'])}</div>"
        f"<ul>{reasons}</ul><div>{html.escape(r.get('next_step', ''))}</div>"
        f"<small>This project is a prototype and is not medical advice.</small></div>"))

## 2. The data
We wrote **60 short instructions in the style of medicine labels** (doses, how often, limits, warnings, storage, ...). They are **synthetic**: the drug names are made up and nothing is copied from a real label, so no real patient or prescription text is used. We split them by sentence into **30 "dev"** sentences (used only to choose thresholds) and **30 "test"** sentences (used for every number we report). A sentence has the same split in all three languages, so nothing leaks from dev to test.

In [ ]:
# 2. DATA - our 60 synthetic label-style sentences.
sentences = pd.read_csv("data/sentences_en.csv", encoding="utf-8-sig", dtype=str)
print(f"TIE: sentences loaded = {len(sentences)}, expected 60, Diff = {len(sentences) - 60}")
print("dev/test split:", sentences["split"].value_counts().to_dict())
display(sentences.groupby(["category", "split"]).size().unstack(fill_value=0))
display(sentences[["sentence_id", "category", "split", "src_en"]].head(6))

## 3. How often is the machine translation unsafe? (human ratings)
We translated every sentence into Hindi, Chinese and Arabic with the free **NLLB-200 (distilled, 600M)** model. Then **each of us, a speaker of one language, rated all 60 translations by hand** without seeing any automatic score:

- **safe** — same instruction, same numbers, same warnings (awkward wording is fine)
- **minor** — odd or wrong wording, but a reader would still do the right thing
- **unsafe** — a reader could do something different from the English (changed number or unit, lost "do not", wrong drug name, dropped or invented instruction, fluent but different meaning)

If we were unsure between minor and unsafe we chose *unsafe* (safety first). Ratings were made by humans only.

In [ ]:
# 3. HUMAN RATINGS - how often did a speaker judge the machine translation unsafe?
by_lang = load_csv("results_by_lang.csv")
master = load_csv("master.csv")

if by_lang is None and master is not None:                       # the results file is missing: count ourselves
    counts = master.groupby(["lang", "verdict"]).size().unstack(fill_value=0).reset_index()
    for v in ("safe", "minor", "unsafe"):
        if v not in counts:
            counts[v] = 0
    counts["n"] = counts[["safe", "minor", "unsafe"]].sum(axis=1)
    counts["unsafe_rate"] = counts["unsafe"] / counts["n"]
    by_lang = counts

if by_lang is not None:
    by_lang = num(by_lang, ["n", "safe", "minor", "unsafe", "unsafe_rate", "unsafe_lo", "unsafe_hi"])
    shown = [c for c in ["lang", "n", "safe", "minor", "unsafe", "unsafe_rate", "unsafe_lo", "unsafe_hi"] if c in by_lang]
    display(by_lang[shown].round(3))

    chart = by_lang[by_lang["lang"].isin(LANGS)].set_index("lang").reindex(LANGS).dropna(subset=["n"])
    names = [LANG_NAMES[l] for l in chart.index]
    fig, ax = plt.subplots(figsize=(6.5, 3.8))
    bottom = np.zeros(len(chart))
    for verdict, color in [("safe", "#4A7FB5"), ("minor", "#E0B33A"), ("unsafe", "#C8453B")]:
        vals = chart[verdict].to_numpy(dtype=float)
        ax.bar(names, vals, bottom=bottom, color=color, label=verdict)
        for i, v in enumerate(vals):
            if v > 0:
                ax.text(i, bottom[i] + v / 2, f"{int(v)}", ha="center", va="center", fontsize=11,
                        color="black" if verdict == "minor" else "white")
        bottom += vals
    ax.set_ylabel("translations rated")
    ax.set_title("Speaker ratings of the NLLB-200 translations")
    ax.legend(title="rating", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
    plt.tight_layout()
    plt.show()

etypes = load_csv("results_error_types.csv")
if etypes is not None:
    etypes = num(etypes, ["count"])
    display(etypes.pivot_table(index="error_type", columns="lang", values="count", aggfunc="sum", fill_value=0))
elif master is not None:
    display(master[master["verdict"] != "safe"].groupby(["error_type", "lang"]).size().unstack(fill_value=0))

**Read-out (H1):** <<A: 1-2 sentences with counts — which language had the most unsafe translations, which error type was most common, and whether the rate differs between languages.>>

## 4. Can a warning catch the unsafe ones? (detectors vs the do-nothing baseline)
On the **30 test sentences per language** (never used to choose anything) we compare ways of warning the reader:

- **Always says "fine"** — the do-nothing baseline. It is right most of the time and catches nothing.
- **Round-trip chrF score** — translate back to English and measure how similar it is to the original. A normal quality score. The cut-off was chosen on the dev sentences only.
- **Number check** — do the same numbers appear in the English and the translation?
- **Negation check** — if the English says "do not / avoid / without", does the translation still have a "not" word?
- **Combinations** of these.

Because unsafe translations are the rare event we care about, we report **recall** (how many unsafe ones were flagged) and **precision / false alarms**, not accuracy.

In [ ]:
# 4. DETECTORS - on the TEST sentences only, how many unsafe translations does each method flag?
det = load_csv("results_detectors.csv")
if det is not None:
    det = num(det, ["n_test", "tp", "fp", "fn", "tn", "precision", "recall", "f1", "flag_rate",
                    "false_alarms_per_10_nonunsafe", "precision_lo", "precision_hi", "recall_lo", "recall_hi"])
    det["method"] = det["detector"].map(DETECTOR_NAMES).fillna(det["detector"])

    # Control total: in every row the four counts must add up to the number of test rows.
    off = int((det[["tp", "fp", "fn", "tn"]].sum(axis=1) != det["n_test"]).sum())
    print(f"{'TIE' if off == 0 else 'GAP'}: tp+fp+fn+tn = test rows for all {len(det)} rows, Diff = {off}")

    pooled = det[det["lang"] == "all"].copy()
    display(pooled[["method", "tp", "fn", "fp", "tn", "recall", "precision", "false_alarms_per_10_nonunsafe"]].round(2))

    # Chart: recall with its 95% interval; the text on the right says "caught of total unsafe".
    rec = np.nan_to_num(pooled["recall"].to_numpy(dtype=float), nan=0.0)
    lo = np.nan_to_num(pooled["recall_lo"].to_numpy(dtype=float), nan=0.0) if "recall_lo" in pooled else rec
    hi = np.nan_to_num(pooled["recall_hi"].to_numpy(dtype=float), nan=0.0) if "recall_hi" in pooled else rec
    err = np.vstack([np.clip(rec - lo, 0, None), np.clip(hi - rec, 0, None)])
    y = np.arange(len(pooled))
    fig, ax = plt.subplots(figsize=(8.5, 0.6 * len(pooled) + 1.6))
    ax.barh(y, rec, xerr=err, color="#4A7FB5", ecolor="#444444", capsize=3)
    for i, (tp, fn) in enumerate(zip(pooled["tp"], pooled["fn"])):
        ax.text(1.02, i, f"{int(tp)} of {int(tp + fn)}", va="center")
    ax.set_yticks(y)
    ax.set_yticklabels(pooled["method"])
    ax.invert_yaxis()
    ax.set_xlim(0, 1)
    ax.set_xlabel("Recall: share of unsafe test translations that were flagged (bars: 95% interval)")
    plt.tight_layout()
    plt.show()

    # Same numbers per language ("caught of total unsafe").
    det["caught"] = (det["tp"].fillna(0).astype(int).astype(str) + " of "
                     + (det["tp"] + det["fn"]).fillna(0).astype(int).astype(str))
    present = [c for c in LANGS if c in set(det["lang"])]
    display(det.pivot(index="method", columns="lang", values="caught")[present])

**Read-out (H2, H3):** <<A: say in plain words — did "always fine" catch 0? How did the chrF score compare with the number/negation checks (recall, false alarms per 10)? Which hypotheses held and which did not? Mention how wide the intervals are.>>

## 5. Where it fails
Required by the track: show the machine output next to the correct output. **FN** = an unsafe translation our best detector missed. **FP** = a translation a speaker judged fine that we flagged anyway.

In [ ]:
# 5. FAILURES - side by side: English / machine translation / correct translation.
fails = load_csv("failures.csv")
print("Unsafe translations our checks MISSED (false negatives):")
show_examples(fails, per_lang=3, kind="FN")
print("Fine translations we FLAGGED anyway (false alarms):")
show_examples(fails, per_lang=2, kind="FP")

**Why we think these happened:**
- <<hi: one sentence per example — e.g. "the translation is fluent but says a different thing, so no number or 'not' word changed">>
- <<zh: ...>>
- <<ar: ...>>

## 6. Does the quality score separate unsafe from fine?
Each dot is one translation. If the round-trip chrF score were a good warning, the "unsafe" dots would sit clearly lower than the "safe" ones.

In [ ]:
# 6. QUALITY SCORE VS HUMAN VERDICT
corr = load_csv("results_correlation.csv")
if corr is not None:
    display(num(corr, ["spearman_rho", "p_value", "n"]).round(3))

feat = load_csv("features.csv")
if master is not None and feat is not None:
    joined = master.merge(feat[["sentence_id", "lang", "system", "rt_chrf"]], on=["sentence_id", "lang", "system"])
    joined["rt_chrf"] = pd.to_numeric(joined["rt_chrf"], errors="coerce")
    rng = np.random.default_rng(0)                                   # fixed seed: same dot positions every run
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True)
    for ax, lang in zip(axes, LANGS):
        sub = joined[joined["lang"] == lang]
        groups = [sub.loc[sub["verdict"] == v, "rt_chrf"].dropna().to_numpy() for v in ("safe", "minor", "unsafe")]
        ax.boxplot(groups, showfliers=False)
        for i, g in enumerate(groups, start=1):
            ax.scatter(i + rng.uniform(-0.15, 0.15, len(g)), g, s=14, alpha=0.6, color="#4A7FB5", zorder=3)
        ax.set_xticks([1, 2, 3])
        ax.set_xticklabels([f"{v}\n(n={len(g)})" for v, g in zip(("safe", "minor", "unsafe"), groups)])
        ax.set_title(LANG_NAMES[lang])
    axes[0].set_ylabel("round-trip chrF (0-100)")
    plt.tight_layout()
    plt.show()

**Read-out:** <<A: is there a visible gap? What does the Spearman correlation say? (Spearman = how well the score ranks translations in the same order as the human verdict; 0 means no relationship.)>>

## 7. Demo
**Who it is for:** a family member at the pharmacy counter who reads the translation, not the English. **Input:** one English instruction and a language. **Output:** the translation plus a verdict and the reasons. We never say a translation is "safe" — the best answer is "no problems found by our checks, still confirm with your pharmacist".

The first cell shows saved example results (works with no GPU and no internet). The second starts the live app.

In [ ]:
# 7a. SAVED DEMO EXAMPLES - results of check_translation() on six chosen sentences.
try:
    from safelabel.predict import check_translation          # H builds this; the notebook still runs without it
except Exception as err:
    check_translation = None
    print("Live checker not available:", err)

cache_file = Path(OUTPUTS_DIR) / "demo_cache.json"
DEMO_EXAMPLES = [
    # <<H: after G3 choose 6 (two per language) from outputs/failures.csv and from clean rows.
    #     Format: ("English sentence", "hi")  where the language is "hi", "zh" or "ar">>
]
REFRESH_CACHE = False     # set True ONCE in a session where the NLLB model loads (GPU), run the cell, then set False

if REFRESH_CACHE and check_translation is not None and DEMO_EXAMPLES:
    cache = [check_translation(src, lang) for src, lang in DEMO_EXAMPLES]
    cache_file.write_text(json.dumps(cache, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Saved", len(cache), "examples to", cache_file)

if cache_file.exists():
    for result in json.loads(cache_file.read_text(encoding="utf-8")):
        show_result(result)
else:
    print("(no saved examples yet: fill DEMO_EXAMPLES, then refresh the cache)")

In [ ]:
# 7b. LIVE APP - paste any English sentence. Needs src/safelabel/predict.py and app/app.py (built by H).
# A GPU makes it faster; without the model it falls back to the saved examples above.
try:
    from app.app import build_demo
    demo = build_demo()
    demo.launch()          # shows the app under this cell; does not block "Run all"
except Exception as err:
    print("Live app not available:", err)

## 8. How we produced the numbers
1. `python src/translate.py --lang <hi|zh|ar>` (Colab GPU) → `outputs/translations_<lang>.csv` — NLLB-200 distilled 600M, beam search 4, plus a back-translation to English.
2. Each of us rated our language's 60 translations by hand in a spreadsheet → `outputs/ratings_<lang>.csv`.
3. `python tools/tie_out.py --all --write` → `outputs/master.csv`. Every control total must print `TIE: ... Diff = 0`.
4. `python src/run_checks.py` → `outputs/features.csv` (number check, negation check, round-trip chrF).
5. `python src/evaluate.py` → result tables, failures, figures. **Thresholds were chosen on the dev sentences only; every detector number above is from the test sentences.**

Unit tests for the checks: `python -m unittest discover -s tests -v`. The full study design is written down in `CONTRACT.md` (hypotheses H1–H3 were fixed before we saw results).

## 9. Limits and what we'd do next
**Limits**
- Small sample: <<n>> test sentences per language and <<n>> unsafe test translations in total, so the intervals are wide. We show counts, not just percentages.
- The raters are speakers of the language but not certified translators. <<second reader, if done>>
- The sentences are synthetic label-style text, not real labels. <<Simplified vs Traditional Chinese; Modern Standard Arabic; Hindi in Devanagari>>
- Our checks only catch changed or missing numbers and lost "not" words. Fluent-but-wrong translations slip through (see section 5).

**Next steps**
- <<e.g. review by certified medical translators and a second human reader on every row>>
- <<e.g. more languages and spoken dialects; real public-domain label text; test other translators>>

> This project is a prototype and is not medical advice.

## 10. Credits
- **Translator:** NLLB-200 distilled 600M — https://huggingface.co/facebook/nllb-200-distilled-600M
- **Libraries:** Hugging Face Transformers — https://github.com/huggingface/transformers · sacreBLEU — https://github.com/mjpost/sacrebleu · Gradio — https://www.gradio.app · pandas, NumPy, SciPy, Matplotlib
- **Data:** 60 synthetic sentences written by our team for this project
- **Related studies:** <<links, each checked by us before quoting>>
- **AI tools:** <<Claude, ChatGPT/Codex, Gemini (if used): what each was used for. All ratings were made by humans.>>